In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. PyTorch Synthetic Dataset (Single-Lead Input -> 12-Lead Target)
class SingleLeadECGDataset(Dataset):
    """
    Input: Tensor of shape (1, seq_len) -> Lead I
    Target: Tensor of shape (12, seq_len) -> Full 12-Lead Array
    """
    def __init__(self, num_samples=128, seq_len=1000, sampling_rate=500):
        super().__init__()
        self.num_samples = num_samples
        self.seq_len = seq_len
        t = np.linspace(0, seq_len / sampling_rate, seq_len)
        data = []
        for i in range(num_samples):
            freq = np.random.uniform(1.0, 1.5) # Heart rate
            qrs = np.sin(2 * np.pi * freq * t) ** 21
            p_wave = 0.18 * np.sin(2 * np.pi * freq * t - 0.5) ** 5
            t_wave = 0.35 * np.sin(2 * np.pi * freq * t + 0.8) ** 3
            
            leads = [amp * (qrs + p_wave + t_wave) for amp in np.linspace(0.4, 1.2, 12)]
            data.append(np.array(leads))
        self.data = torch.tensor(np.array(data), dtype=torch.float32)

    def __len__(self): 
        return self.num_samples

    def __getitem__(self, idx): 
        # Extract Lead I as 1-channel input: shape (1, seq_len)
        input_lead_I = self.data[idx][[0], :] 
        full_12_lead = self.data[idx]
        return input_lead_I, full_12_lead

# 2. 1D U-Net Model (1 Input Channel -> 12 Output Channels)
class SingleLeadUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=12):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            nn.Conv1d(32, 64, kernel_size=15, padding=7),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2)
        )
        self.decoder = nn.Sequential(
            nn.Conv1d(64, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            nn.Conv1d(32, out_channels, kernel_size=1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.decoder(feat)

# 3. Instantiate and Execute Single-Lead Training Loop
dataset = SingleLeadECGDataset(num_samples=128, seq_len=1000)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

model = SingleLeadUNet(in_channels=1, out_channels=12)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

print("=== Executing Single-Lead PyTorch Training Pass ===")
model.train()
for epoch in range(1, 4):
    running_loss = 0.0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        pred = model(x_batch)  # Input (B, 1, 1000) -> Output (B, 12, 1000)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch [{epoch}/3] Training Loss: {running_loss / len(loader):.4f}")

=== Executing Single-Lead PyTorch Training Pass ===


Epoch [1/3] Training Loss: 0.1136


Epoch [2/3] Training Loss: 0.0270


Epoch [3/3] Training Loss: 0.0133


In [3]:
lead_names = ['Lead I (Input)', 'Lead II', 'Lead III', 'aVR', 'aVL', 'aVF', 
              'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

fig_12lead = make_subplots(rows=6, cols=2, subplot_titles=lead_names, shared_xaxes=True)

model.eval()
with torch.no_grad():
    x_test, y_test = dataset[0]
    pred_test = model(x_test.unsqueeze(0)).squeeze(0).numpy()
    target_test = y_test.numpy()

t_axis = np.linspace(0, 2.0, 1000)

for idx, name in enumerate(lead_names):
    r_idx = (idx // 2) + 1
    c_idx = (idx % 2) + 1
    
    # Target Trace (Blue)
    fig_12lead.add_trace(go.Scatter(
        x=t_axis, y=target_test[idx], mode='lines', 
        line=dict(color='#38bdf8', width=1.5), name=f'{name} Target', showlegend=False
    ), row=r_idx, col=c_idx)
    
    # Reconstructed Trace (Red Dash)
    fig_12lead.add_trace(go.Scatter(
        x=t_axis, y=pred_test[idx], mode='lines', 
        line=dict(color='#f43f5e', width=1.5, dash='dash'), name=f'{name} Pred', showlegend=False
    ), row=r_idx, col=c_idx)

fig_12lead.update_layout(
    title="Single-Lead Reconstructed 12-Lead Array vs. Ground Truth Target",
    template="plotly_dark",
    height=900,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_12lead.show()